In [1]:
from dotenv import load_dotenv
from pathlib import Path
import sys
import os

# Walk up until we find the project root (folder with the .env)
current_path = Path().resolve()
for parent in [current_path] + list(current_path.parents):
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        project_root = os.getenv("PROJECT_ROOT")
        print(project_root)
        sys.path.append(project_root)     
        break


%load_ext autoreload
%autoreload 2

Failed to read module file 'C:\Users\Admin\AppData\Roaming\uv\python\cpython-3.12.8-windows-x86_64-none\Lib\shlex.py' for module 'shlex': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Admin\AppData\Roaming\uv\python\cpython-3.12.8-windows-x86_64-none\Lib\importlib\__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootst

C:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\


In [3]:
import pandas as pd
import glob
import os

# Path to the snippets directory
snippets_dir = "../../data/evaluation_snippets/irene/"

# Get all folders in the snippets directory
snippet_folders = [f for f in os.listdir(snippets_dir) 
                   if os.path.isdir(os.path.join(snippets_dir, f))]

# Read all files from each folder's manual_verification directory
dfs = []
for folder in snippet_folders:
    manual_verification_dir = os.path.join(snippets_dir, folder, "manual_verification")
    
    # Check if manual_verification directory exists
    if not os.path.exists(manual_verification_dir):
        print(f"No manual_verification folder found in {folder}")
        continue
    
    # Get all files in the manual_verification directory
    manual_files = glob.glob(os.path.join(manual_verification_dir, "*"))
    
    # Read each file and add folder name as a column
    for file in manual_files:
        try:
            df = pd.read_csv(file, sep='\t', engine='python')
            df['site'] = folder  # Add folder name as a column
            
            df["labeled_snippet_dir"] = os.path.join(snippets_dir, folder)

            # Extract annotator from filename: text between '.selections' and '.txt', remove leading character
            base = os.path.basename(file)
            if '.selections' in base and base.endswith('.txt'):
                annotator_raw = base.split('.selections')[1].replace('.txt', '')
                annotator = annotator_raw[1:] if len(annotator_raw) > 1 else ''
                df['annotator'] = annotator
                
            dfs.append(df)
            print(f"Read {os.path.basename(file)} from {folder}")
        except Exception as e:
            print(f"Could not read {file}: {e}")

# Concatenate all DataFrames into one
if dfs:
    all_manual_df = pd.concat(dfs, ignore_index=True)
    print(f"\nTotal rows: {len(all_manual_df)}")
    print(f"Datasets: {all_manual_df['site'].unique()}")
else:
    print("No data files found")
    all_manual_df = pd.DataFrame()

Read 6871.220821133604.snippet.selections_IR.txt from RDL_2022
Read 6871.220829131406.snippet.selections_IR.txt from RDL_2022
Read 6871.220829131706.snippet.selections_IR.txt from RDL_2022

Total rows: 690
Datasets: <StringArray>
['RDL_2022']
Length: 1, dtype: str


In [4]:
labels_df = all_manual_df.copy()

labels_df = labels_df.drop(columns=["Selection", "View", "Channel", "Low Freq (Hz)", "High Freq (Hz)", "ECHO", "HFPC", "BBPC", "Whistle"])


In [5]:
labels_df["annotator"].value_counts(dropna=False)

annotator
IR    690
Name: count, dtype: int64

In [8]:
labels_df.groupby("annotator")["GROUNDTRUTH"].value_counts(dropna=False)

annotator  GROUNDTRUTH
IR         a              158
           b               64
           e               56
           w               17
           bw              11
           h                8
           hw               4
           hb               3
           wb               1
           eb               1
           eh               1
Name: count, dtype: int64

In [7]:
labels_df = labels_df.dropna(subset=["GROUNDTRUTH"])

In [9]:
labels_df = labels_df.rename(columns={"snippet_filename": "labeled_snippet_filename"})

In [10]:
from pipeline.pipeline import get_hydrophone_model
from data_preprocessing.spectrogram.spectrogram_generator import HYDROPHONE_SENSITIVITY

# Apply get_hydrophone_model to the original_filename column for all rows
labels_df["HydrophoneModel"] = labels_df["original_filename"].apply(get_hydrophone_model)
labels_df["HydrophoneSensitivity"] = labels_df["HydrophoneModel"].apply(HYDROPHONE_SENSITIVITY.get_sensitivity)

In [11]:
labels_df["HydrophoneSensitivity"].value_counts()

HydrophoneSensitivity
-176.3    324
Name: count, dtype: int64

In [12]:
labels_df.tail()

,Begin Time (s),End Time (s),GROUNDTRUTH,DETAILS,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Boat,site,labeled_snippet_dir,annotator,HydrophoneModel,HydrophoneSensitivity
661,121.0,122.0,e,NaN,ship,6871.220829124106.wav,6871.220829131706.snippet.wav,2022-08-29 13:17:06,2160,2310,1,RDL_2022,../../data/evaluation_snippets/irene/RDL_2022,IR,6871,-176.3
662,122.0,123.0,e,NaN,ship,6871.220829124106.wav,6871.220829131706.snippet.wav,2022-08-29 13:17:06,2160,2310,1,RDL_2022,../../data/evaluation_snippets/irene/RDL_2022,IR,6871,-176.3
663,123.0,124.0,e,NaN,ship,6871.220829124106.wav,6871.220829131706.snippet.wav,2022-08-29 13:17:06,2160,2310,1,RDL_2022,../../data/evaluation_snippets/irene/RDL_2022,IR,6871,-176.3
664,124.0,125.0,b,b,ship,6871.220829124106.wav,6871.220829131706.snippet.wav,2022-08-29 13:17:06,2160,2310,1,RDL_2022,../../data/evaluation_snippets/irene/RDL_2022,IR,6871,-176.3
665,125.0,126.0,e,NaN,ship,6871.220829124106.wav,6871.220829131706.snippet.wav,2022-08-29 13:17:06,2160,2310,1,RDL_2022,../../data/evaluation_snippets/irene/RDL_2022,IR,6871,-176.3


In [13]:
import pandas as pd

# Convert snippet_start_time to datetime and add the offset in seconds
labels_df["clip_start_time"] = pd.to_datetime(labels_df["snippet_start_time"]) + pd.to_timedelta(labels_df["Begin Time (s)"], unit='s')
labels_df["clip_end_time"] = pd.to_datetime(labels_df["snippet_start_time"]) + pd.to_timedelta(labels_df["End Time (s)"], unit='s')

In [14]:
labels_df.rename(columns={"site": "Site"}, inplace=True)
labels_df["Site"] = labels_df["Site"].str.split("_").str[0]

In [20]:
pd.set_option('display.max_columns', None)
labels_df["clip_filename"] = labels_df["Site"] + "_" + labels_df["clip_start_time"].dt.strftime("%Y%m%d_%H%M%S%f").str[:-4] + ".wav"
# Check for duplicates in clip_filename
duplicate_clips = labels_df[labels_df.duplicated("clip_filename", keep=False)]
if not duplicate_clips.empty:
    print("Duplicates found in 'clip_filename':")
    print(len(duplicate_clips))
    print(duplicate_clips["original_filename"].value_counts())
    display(duplicate_clips.sort_values(by="clip_filename").head(3))
else:
    print("No duplicates found in 'clip_filename'.")

Duplicates found in 'clip_filename':
36
original_filename
6871.220829124106.wav    36
Name: count, dtype: int64


,Begin Time (s),End Time (s),GROUNDTRUTH,DETAILS,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Boat,Site,labeled_snippet_dir,annotator,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,clip_filename
367,187.0,188.0,h,NaN,ship,6871.220829124106.wav,6871.220829131406.snippet.wav,2022-08-29 13:14:06,1980,2340,1,RDL,../../data/evaluation_snippets/irene/RDL_2022,IR,6871,-176.3,2022-08-29 13:17:13,2022-08-29 13:17:14,RDL_20220829_13171300.wav
547,7.0,8.0,h,NaN,ship,6871.220829124106.wav,6871.220829131706.snippet.wav,2022-08-29 13:17:06,2160,2310,1,RDL,../../data/evaluation_snippets/irene/RDL_2022,IR,6871,-176.3,2022-08-29 13:17:13,2022-08-29 13:17:14,RDL_20220829_13171300.wav
368,188.0,189.0,h,NaN,ship,6871.220829124106.wav,6871.220829131406.snippet.wav,2022-08-29 13:14:06,1980,2340,1,RDL,../../data/evaluation_snippets/irene/RDL_2022,IR,6871,-176.3,2022-08-29 13:17:14,2022-08-29 13:17:15,RDL_20220829_13171400.wav


In [23]:
labels_df.drop_duplicates(subset=["clip_filename"], inplace=True)

6871.220829124106 : 35/36- 38.5
6871.220821132704 : 2-4
6871.220821132704 : 9-12
6871.220829124106 : 33-39

In [24]:
def set_verif_flags(gt):
    if pd.isna(gt):
        return pd.Series([False, False, False, False])
    gt_str = str(gt)
    if 'a' in gt_str:
        return pd.Series([False, False, False, False])
    return pd.Series([
        'e' in gt_str,  # ECHO_verif
        'b' in gt_str,  # BBPC_verif
        'h' in gt_str,  # HFPC_verif
        'w' in gt_str   # Whislte_verif
    ])

labels_df[["ECHO", "BBPC", "HFPC", "Whistle"]] = labels_df["GROUNDTRUTH"].apply(set_verif_flags)
# Convert ECHO, BBPC, HFPC, Whistle columns to 0/1 integers
labels_df[["ECHO", "BBPC", "HFPC", "Whistle"]] = labels_df[["ECHO", "BBPC", "HFPC", "Whistle"]].astype(int)


In [25]:
labels_df["BBPC"].value_counts()

BBPC
0    232
1     74
Name: count, dtype: int64

## Clipping to 1 second audio files

In [26]:
import os
import librosa
import soundfile as sf
from tqdm import tqdm

# Output directory
output_dir = "../../data/Verified_Dataset/clip_wavs"
os.makedirs(output_dir, exist_ok=True)

# Group by snippet to load each file only once
grouped = labels_df.groupby(["labeled_snippet_dir", "labeled_snippet_filename"])

for (snippet_dir, snippet_filename), group in tqdm(grouped, total=len(grouped)):
    # Build the full path to the source snippet
    source_path = os.path.join(snippet_dir, snippet_filename)
    
    try:
        # Load the entire snippet once
        y, sr = librosa.load(source_path, sr=None)
        
        # Extract all clips from this snippet
        for idx, row in group.iterrows():
            output_path = os.path.join(output_dir, row["clip_filename"])
            
            # Skip if already exists
            if os.path.exists(output_path):
                continue
            
            # Calculate sample indices
            start_sample = int(row["Begin Time (s)"] * sr)
            end_sample = int(row["End Time (s)"] * sr)
            
            # Extract and save the clip
            clip = y[start_sample:end_sample]
            sf.write(output_path, clip, sr)
            
    except Exception as e:
        print(f"Error processing {snippet_filename}: {e}")

  0%|          | 0/3 [00:00<?, ?it/s]c:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 3/3 [00:02<00:00,  1.06it/s]


In [28]:
labels_df["Boat"].value_counts()

Boat
1    306
Name: count, dtype: int64

In [31]:
labels_df.groupby("labeled_snippet_filename")["BBPC"].value_counts()

labeled_snippet_filename       BBPC
6871.220821133604.snippet.wav  0       112
                               1        68
6871.220829131406.snippet.wav  0        84
                               1         6
6871.220829131706.snippet.wav  0        36
Name: count, dtype: int64

In [30]:
labels_output_dir = "../../data/Verified_Dataset/labels"

labels_df.to_csv(os.path.join(labels_output_dir, "labels_irene_rdl.csv"), index=False)